# AI-Driven Autonomous DSS for SME Analytics — Experiments

**Research Report Companion Notebook**
*Designing an AI-Driven Autonomous Decision Support System for Small and Medium Enterprise Analytics*

This notebook implements the three analytical streams described in Chapter 3 of the research report:
1. **Customer Segmentation** — k-Means clustering on RFM features
2. **Sales Forecasting** — Random Forest, Gradient Boosting, and LSTM
3. **Anomaly Detection** — Isolation Forest on transactional features

**Dataset:** UCI Online Retail II (1.07M transactions from a UK-based SME, 2009–2011)
**Source:** https://archive.ics.uci.edu/dataset/502/online+retail+ii
**License:** Creative Commons Attribution 4.0 International (CC BY 4.0)

---

## How to run this notebook
1. Upload `online_retail_II.xlsx` to the Colab files panel (or mount Google Drive).
2. Run all cells top-to-bottom. The full pipeline takes ~5–10 minutes on a free Colab CPU instance, ~2 minutes on a T4 GPU.
3. All outputs (cluster profiles, model metrics, trained models) are saved to `./outputs/` for the Streamlit app to consume.


## 1. Environment Setup

In [ ]:
# Install missing packages (Colab usually has most)
!pip install -q openpyxl tensorflow scikit-learn pandas numpy matplotlib seaborn joblib


In [ ]:
import os, warnings, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)
np.random.seed(42)

print('Environment ready.')


## 2. Load and Inspect the Dataset

The dataset is provided as an Excel file with two sheets (one per year). We concatenate them into a single DataFrame.


In [ ]:
# Path to the dataset — adjust if needed
DATA_PATH = 'online_retail_II.xlsx'

xls = pd.ExcelFile(DATA_PATH)
print('Sheets found:', xls.sheet_names)

df_list = [pd.read_excel(xls, sheet_name=s) for s in xls.sheet_names]
df = pd.concat(df_list, ignore_index=True)

print(f'\nTotal records: {len(df):,}')
print(f'Date range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
print(f'\nColumns: {df.columns.tolist()}')
df.head()


In [ ]:
# Quick descriptive stats
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nUnique customers: {df["Customer ID"].nunique():,}')
print(f'Unique products: {df["StockCode"].nunique():,}')
print(f'Unique countries: {df["Country"].nunique()}')


## 3. Data Pre-Processing

Following Section 3.2.3 of the methodology:
- Remove records with missing `Customer ID`
- Remove non-positive prices and quantities
- Separate cancellations (Invoice prefix `C`) for anomaly analysis
- Compute `TotalRevenue = Quantity × Price`
- Extract temporal features


In [ ]:
# Normalise column names for easier handling
df.rename(columns={'Customer ID': 'CustomerID', 'Invoice': 'InvoiceNo'}, inplace=True)

# Flag cancellations BEFORE filtering
df['IsCancellation'] = df['InvoiceNo'].astype(str).str.startswith('C')

# Keep cancellations separately for anomaly analysis
cancellations = df[df['IsCancellation']].copy()
print(f'Cancellation records (saved separately): {len(cancellations):,}')

# Clean main dataframe
df_clean = df[~df['IsCancellation']].copy()
df_clean = df_clean.dropna(subset=['CustomerID'])
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# Compute revenue
df_clean['TotalRevenue'] = df_clean['Quantity'] * df_clean['Price']

# Temporal features
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Date'] = df_clean['InvoiceDate'].dt.date
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Quarter'] = df_clean['InvoiceDate'].dt.quarter

print(f'Clean records: {len(df_clean):,}')
print(f'Total revenue: £{df_clean["TotalRevenue"].sum():,.2f}')
df_clean.head()


## 4. Exploratory Data Analysis

In [ ]:
# Daily revenue trend
daily = df_clean.groupby('Date').agg(
    Revenue=('TotalRevenue', 'sum'),
    Transactions=('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).reset_index()
daily['Date'] = pd.to_datetime(daily['Date'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily['Date'], daily['Revenue'], linewidth=0.8, color='#1a3a5c')
ax.set_title('Daily Revenue Over Time')
ax.set_xlabel('Date'); ax.set_ylabel('Revenue (£)')
plt.tight_layout()
plt.savefig('outputs/figures/daily_revenue.png', dpi=120, bbox_inches='tight')
plt.show()

# Top countries
top_countries = df_clean.groupby('Country')['TotalRevenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 4))
top_countries.plot(kind='barh', ax=ax, color='#1a6b6b')
ax.set_title('Top 10 Countries by Revenue')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('outputs/figures/top_countries.png', dpi=120, bbox_inches='tight')
plt.show()


## 5. Stream 1 — Customer Segmentation (k-Means on RFM)

We compute Recency–Frequency–Monetary (RFM) features per customer, standardise them, and apply k-Means.
The optimal **k** is selected via Elbow + Silhouette analysis.


In [ ]:
# Build RFM features
snapshot_date = df_clean['InvoiceDate'].max() + timedelta(days=1)

rfm = df_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalRevenue', 'sum')
).reset_index()

# Remove pathological outliers (top 1% Monetary)
rfm = rfm[rfm['Monetary'] < rfm['Monetary'].quantile(0.99)]
print(f'Customers after outlier trim: {len(rfm):,}')
rfm.describe().round(2)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Log-transform Monetary and Frequency (heavily skewed), then standardise
rfm_log = rfm.copy()
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary'] = np.log1p(rfm_log['Monetary'])

scaler = StandardScaler()
X = scaler.fit_transform(rfm_log[['Recency', 'Frequency', 'Monetary']])

# Elbow + Silhouette
inertia, sil_scores, db_scores = [], [], []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))
    db_scores.append(davies_bouldin_score(X, labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(k_range, inertia, 'o-', color='#1a3a5c'); axes[0].set_title('Elbow Method'); axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[1].plot(k_range, sil_scores, 'o-', color='#1a6b3c'); axes[1].set_title('Silhouette Score (higher = better)'); axes[1].set_xlabel('k')
axes[2].plot(k_range, db_scores, 'o-', color='#8b1a1a'); axes[2].set_title('Davies-Bouldin (lower = better)'); axes[2].set_xlabel('k')
plt.tight_layout()
plt.savefig('outputs/figures/cluster_selection.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Fit final model with k=4 (typical choice for RFM)
K_OPTIMAL = 4
final_km = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
rfm['Cluster'] = final_km.fit_predict(X)

# Cluster profiles (in original units, not log-scaled)
profile = rfm.groupby('Cluster').agg(
    Customers=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(2)

# Heuristic naming
def name_cluster(row):
    if row['Avg_Recency'] < 60 and row['Avg_Monetary'] > rfm['Monetary'].median():
        return 'Champions'
    elif row['Avg_Recency'] < 90:
        return 'Loyal Customers'
    elif row['Avg_Recency'] > 180:
        return 'At-Risk / Hibernating'
    else:
        return 'Potential Loyalists'

profile['Segment'] = profile.apply(name_cluster, axis=1)
print('Cluster profiles:')
display(profile)

# Save metrics + model
seg_metrics = {
    'k': K_OPTIMAL,
    'silhouette_score': float(silhouette_score(X, rfm['Cluster'])),
    'davies_bouldin_score': float(davies_bouldin_score(X, rfm['Cluster'])),
    'n_customers': int(len(rfm))
}
joblib.dump(final_km, 'outputs/models/kmeans.joblib')
joblib.dump(scaler, 'outputs/models/rfm_scaler.joblib')
rfm.to_csv('outputs/rfm_segments.csv', index=False)
profile.to_csv('outputs/cluster_profile.csv')
with open('outputs/segmentation_metrics.json', 'w') as f:
    json.dump(seg_metrics, f, indent=2)
print('\nSegmentation metrics:', seg_metrics)


In [ ]:
# 3D scatter — visual sanity check
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
colors = ['#1a3a5c', '#1a6b3c', '#d64f12', '#5a2d82', '#b8860b']
for c in range(K_OPTIMAL):
    sub = rfm[rfm['Cluster'] == c]
    ax.scatter(sub['Recency'], sub['Frequency'], sub['Monetary'],
               c=colors[c], label=f'Cluster {c}', alpha=0.5, s=15)
ax.set_xlabel('Recency (days)'); ax.set_ylabel('Frequency'); ax.set_zlabel('Monetary (£)')
ax.set_title(f'Customer Segments (k={K_OPTIMAL})')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/cluster_3d.png', dpi=120, bbox_inches='tight')
plt.show()


## 6. Stream 2 — Sales Forecasting

We forecast daily revenue using three models:
- **Random Forest Regressor** (ensemble baseline)
- **Gradient Boosting Regressor** (sequential ensemble)
- **LSTM** (deep learning for temporal dependencies)

A chronological 80/20 split prevents future-data leakage.


In [ ]:
# Build daily series
daily_full = df_clean.groupby('Date').agg(Revenue=('TotalRevenue', 'sum')).reset_index()
daily_full['Date'] = pd.to_datetime(daily_full['Date'])
daily_full = daily_full.sort_values('Date').reset_index(drop=True)
daily_full['DayOfWeek'] = daily_full['Date'].dt.dayofweek
daily_full['Month'] = daily_full['Date'].dt.month
daily_full['Quarter'] = daily_full['Date'].dt.quarter
daily_full['DayOfYear'] = daily_full['Date'].dt.dayofyear

# Lag features (using past 7 days)
for lag in [1, 2, 3, 7, 14]:
    daily_full[f'Lag_{lag}'] = daily_full['Revenue'].shift(lag)
daily_full['MA_7'] = daily_full['Revenue'].shift(1).rolling(7).mean()
daily_full['MA_30'] = daily_full['Revenue'].shift(1).rolling(30).mean()

ts = daily_full.dropna().reset_index(drop=True)
print(f'Time-series rows after lag generation: {len(ts)}')
ts.head()


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = ['DayOfWeek', 'Month', 'Quarter', 'DayOfYear',
                'Lag_1', 'Lag_2', 'Lag_3', 'Lag_7', 'Lag_14', 'MA_7', 'MA_30']
X_ts = ts[feature_cols].values
y_ts = ts['Revenue'].values

# Chronological 80/20 split
split = int(len(ts) * 0.8)
X_train, X_test = X_ts[:split], X_ts[split:]
y_train, y_test = y_ts[:split], y_ts[split:]
dates_test = ts['Date'].values[split:]

print(f'Train: {len(X_train)} | Test: {len(X_test)}')


In [ ]:
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name:25s} MAE={mae:>10,.0f}  RMSE={rmse:>10,.0f}  R²={r2:>6.3f}')
    return {'model': name, 'mae': mae, 'rmse': rmse, 'r2': r2}

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_metrics = evaluate(y_test, rf_pred, 'Random Forest')

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_metrics = evaluate(y_test, gb_pred, 'Gradient Boosting')


In [ ]:
# LSTM — use only the lag features + temporal indicators in a sequence form
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

SEQ_LEN = 14
revenue = ts['Revenue'].values.reshape(-1, 1)
rev_scaler = MinMaxScaler()
rev_scaled = rev_scaler.fit_transform(revenue)

X_seq, y_seq = [], []
for i in range(SEQ_LEN, len(rev_scaled)):
    X_seq.append(rev_scaled[i-SEQ_LEN:i, 0])
    y_seq.append(rev_scaled[i, 0])
X_seq, y_seq = np.array(X_seq), np.array(y_seq)
X_seq = X_seq.reshape(-1, SEQ_LEN, 1)

split_l = int(len(X_seq) * 0.8)
Xl_tr, Xl_te = X_seq[:split_l], X_seq[split_l:]
yl_tr, yl_te = y_seq[:split_l], y_seq[split_l:]

tf.random.set_seed(42)
lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm.compile(optimizer='adam', loss='mse')
hist = lstm.fit(Xl_tr, yl_tr, epochs=30, batch_size=32, validation_split=0.1, verbose=0)

lstm_pred_scaled = lstm.predict(Xl_te, verbose=0).flatten()
lstm_pred = rev_scaler.inverse_transform(lstm_pred_scaled.reshape(-1,1)).flatten()
y_te_actual = rev_scaler.inverse_transform(yl_te.reshape(-1,1)).flatten()
lstm_metrics = evaluate(y_te_actual, lstm_pred, 'LSTM')

# Save loss curve
plt.figure(figsize=(10,3))
plt.plot(hist.history['loss'], label='train')
plt.plot(hist.history['val_loss'], label='val')
plt.title('LSTM Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE (scaled)')
plt.legend(); plt.tight_layout()
plt.savefig('outputs/figures/lstm_loss.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Forecast comparison plot
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(dates_test, y_test, label='Actual', color='black', linewidth=1.5)
ax.plot(dates_test, rf_pred, label='Random Forest', alpha=0.7, color='#1a3a5c')
ax.plot(dates_test, gb_pred, label='Gradient Boosting', alpha=0.7, color='#1a6b3c')
ax.set_title('Daily Revenue Forecast — Test Period (RF & GB)')
ax.set_xlabel('Date'); ax.set_ylabel('Revenue (£)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/forecast_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# Save metrics + models
all_metrics = [rf_metrics, gb_metrics, lstm_metrics]
with open('outputs/forecasting_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

joblib.dump(rf, 'outputs/models/rf.joblib')
joblib.dump(gb, 'outputs/models/gb.joblib')
lstm.save('outputs/models/lstm.keras')
joblib.dump(rev_scaler, 'outputs/models/rev_scaler.joblib')
print('\nForecasting models saved.')


## 7. Stream 3 — Anomaly Detection (Isolation Forest)

Detect transaction-level outliers using Isolation Forest on `Quantity`, `Price`, and `TotalRevenue`.
We use a `contamination` rate of 1% (a reasonable prior for transactional fraud/error).


In [ ]:
from sklearn.ensemble import IsolationForest

# Sample for speed (full dataset is ~800k transactions; iForest scales but this saves a few minutes)
SAMPLE_SIZE = 200_000
df_sample = df_clean.sample(n=min(SAMPLE_SIZE, len(df_clean)), random_state=42).copy()

anomaly_features = df_sample[['Quantity', 'Price', 'TotalRevenue']].copy()
# Log transform to compress heavy tails
anomaly_features_log = np.log1p(anomaly_features)

iso = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
df_sample['AnomalyScore'] = iso.fit_predict(anomaly_features_log)
df_sample['AnomalyScoreRaw'] = iso.score_samples(anomaly_features_log)

n_anomalies = (df_sample['AnomalyScore'] == -1).sum()
print(f'Detected anomalies: {n_anomalies:,} out of {len(df_sample):,} ({100*n_anomalies/len(df_sample):.2f}%)')

# Show a few examples
anomalies_view = df_sample[df_sample['AnomalyScore'] == -1].sort_values('AnomalyScoreRaw').head(10)
print('\nTop 10 most anomalous transactions:')
display(anomalies_view[['InvoiceNo','StockCode','Description','Quantity','Price','TotalRevenue','Country']])


In [ ]:
# Visualise anomalies in feature space
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (a, b) in zip(axes, [('Quantity', 'Price'), ('Quantity', 'TotalRevenue'), ('Price', 'TotalRevenue')]):
    normal = df_sample[df_sample['AnomalyScore'] == 1].sample(min(5000, (df_sample['AnomalyScore']==1).sum()))
    anom = df_sample[df_sample['AnomalyScore'] == -1]
    ax.scatter(normal[a], normal[b], s=5, alpha=0.3, label='Normal', color='#1a6b6b')
    ax.scatter(anom[a], anom[b], s=12, alpha=0.7, label='Anomaly', color='#d64f12')
    ax.set_xscale('symlog'); ax.set_yscale('symlog')
    ax.set_xlabel(a); ax.set_ylabel(b); ax.legend()
plt.suptitle('Isolation Forest Detected Anomalies')
plt.tight_layout()
plt.savefig('outputs/figures/anomalies.png', dpi=120, bbox_inches='tight')
plt.show()

# Save model + summary
joblib.dump(iso, 'outputs/models/isolation_forest.joblib')
anomaly_metrics = {
    'sample_size': int(len(df_sample)),
    'n_anomalies_detected': int(n_anomalies),
    'anomaly_rate_pct': float(100*n_anomalies/len(df_sample)),
    'contamination_param': 0.01,
    'cancellation_records_in_dataset': int(len(cancellations))
}
with open('outputs/anomaly_metrics.json', 'w') as f:
    json.dump(anomaly_metrics, f, indent=2)
print('Anomaly metrics:', anomaly_metrics)


## 8. Consolidated Results Summary

In [ ]:
print('='*70)
print('CONSOLIDATED EXPERIMENTAL RESULTS')
print('='*70)

print('\n--- Customer Segmentation (k-Means) ---')
for k, v in seg_metrics.items():
    print(f'  {k:.<35s} {v}')

print('\n--- Sales Forecasting (Daily Revenue) ---')
for m in all_metrics:
    print(f'  {m["model"]:.<25s}  MAE={m["mae"]:>10,.0f}  RMSE={m["rmse"]:>10,.0f}  R²={m["r2"]:>6.3f}')

print('\n--- Anomaly Detection (Isolation Forest) ---')
for k, v in anomaly_metrics.items():
    print(f'  {k:.<35s} {v}')

print('\n' + '='*70)
print('All outputs saved to ./outputs/ directory.')
print('These files are consumed by the Streamlit dashboard (app.py).')
print('='*70)


## 9. Next Steps

1. Download the entire `outputs/` folder.
2. Place it next to `app.py` (the Streamlit dashboard).
3. Run `streamlit run app.py` locally to launch the interactive DSS interface.

The Streamlit prototype demonstrates **Phase 5 of the experimental procedure** (Section 3.7) — integration of the trained models into a non-technical user interface.
